# SPR-03 — Model Training + Ensembling

**Ticket:** SPR-03
**Owner:** Sakhiur
**Depends on:** SPR-02 (resampled train set)
**Folder:** `notebooks/04_modeling/02_ensembling.ipynb`
**Output:** trained models saved to `models/{decision_tree,logistic_regression,random_forest,xgboost}/`, metrics logged to `results/tables/`

**Inspiration:** `SOTA_Paper_8.ipynb` cells 37-48 (hyperparameter search + VotingClassifier hard/soft voting), `cross-validation.ipynb` cells 37/40 (Stratified K-Fold evaluation loop). Adapted for multi-class `income_class` and depth-capped models to avoid the Colab stall documented earlier in the project.

## 1. Load resampled train set from SPR-02

In [ ]:
! wget https://huggingface.co/datasets/Sakhiur/signal/resolve/main/final_dataset_preprocessed_distributed_social_class.csv

In [ ]:
import pandas as pd
import numpy as np
import random
import joblib
import os

np.random.seed(42)
random.seed(42)

X_train = pd.read_csv('data/interim/X_train_resampled.csv')
y_train = pd.read_csv('data/interim/y_train_resampled.csv').iloc[:, 0]
X_test = pd.read_csv('data/interim/X_test.csv')
y_test = pd.read_csv('data/interim/y_test.csv').iloc[:, 0]

print('Train:', X_train.shape, 'Test:', X_test.shape)

## 2. Hyperparameter search space
Same structure as `SOTA_Paper_8.ipynb` cell 39, trimmed to the four models your project actually uses (decision tree, RF, XGBoost, logistic regression) and depth-capped to stay inside Colab free-tier limits, per the compute constraint already flagged for this project.

In [ ]:
param_distributions = {
    'DecisionTreeClassifier': {
        'max_depth': [5, 10, 15, 20],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4]
    },
    'RandomForestClassifier': {
        'n_estimators': [100, 150, 200],
        'max_depth': [10],  # capped per project compute constraint, do not widen
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4]
    },
    'XGBClassifier': {
        'n_estimators': [100, 150, 200],
        'max_depth': [6, 8, 10],
        'learning_rate': [0.01, 0.05, 0.1]
    },
    'LogisticRegression': {
        'C': [0.01, 0.1, 1.0, 10.0],
        'max_iter': [1000]
    }
}

## 3. Randomized search per model

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

base_models = {
    'DecisionTreeClassifier': DecisionTreeClassifier(random_state=42),
    'RandomForestClassifier': RandomForestClassifier(random_state=42),
    'XGBClassifier': XGBClassifier(random_state=42, eval_metric='mlogloss'),
    'LogisticRegression': LogisticRegression(random_state=42)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
best_params = {}
best_estimators = {}

for name, model in base_models.items():
    print(f'Searching {name}...')
    search = RandomizedSearchCV(
        model, param_distributions[name],
        n_iter=10, cv=cv, scoring='f1_macro',
        random_state=42, n_jobs=-1
    )
    search.fit(X_train, y_train)
    best_params[name] = search.best_params_
    best_estimators[name] = search.best_estimator_
    print(f'  Best params: {search.best_params_}')
    print(f'  Best CV macro F1: {search.best_score_:.4f}')

## 4. Evaluate each tuned model on the held-out test set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import seaborn as sns
import matplotlib.pyplot as plt

test_results = {}
for name, model in best_estimators.items():
    y_pred = model.predict(X_test)
    macro_f1 = f1_score(y_test, y_pred, average='macro')
    test_results[name] = macro_f1
    print(f'--- {name} ---')
    print(classification_report(y_test, y_pred, digits=3))
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix: {name}')
    plt.xlabel('Predicted'); plt.ylabel('True')
    plt.show()

## 5. Ensembling: hard + soft voting
Adapted from `SOTA_Paper_8.ipynb` cell 44. `voting='soft'` requires every base estimator to support `predict_proba`, which all four models here do.

In [ ]:
from sklearn.ensemble import VotingClassifier

voting_clf_hard = VotingClassifier(
    estimators=[(name, model) for name, model in best_estimators.items()],
    voting='hard'
)
voting_clf_soft = VotingClassifier(
    estimators=[(name, model) for name, model in best_estimators.items()],
    voting='soft'
)

print('Training hard voting ensemble...')
voting_clf_hard.fit(X_train, y_train)
print('Training soft voting ensemble...')
voting_clf_soft.fit(X_train, y_train)

## 6. Evaluate ensembles

In [ ]:
for clf, label in zip([voting_clf_hard, voting_clf_soft], ['Hard Voting', 'Soft Voting']):
    y_pred = clf.predict(X_test)
    macro_f1 = f1_score(y_test, y_pred, average='macro')
    test_results[label] = macro_f1
    print(f'--- {label} ---')
    print(classification_report(y_test, y_pred, digits=3))
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix: {label}')
    plt.xlabel('Predicted'); plt.ylabel('True')
    plt.show()

print('\nFinal comparison (macro F1):')
for name, score in sorted(test_results.items(), key=lambda x: -x[1]):
    print(f'{name}: {score:.4f}')

## 7. Save models and metrics
This is the handoff point for SPR-04 (figures) and SPR-05 (SHAP), both of which read these saved models read-only and must not retrain them.

In [ ]:
os.makedirs('models/decision_tree', exist_ok=True)
os.makedirs('models/random_forest', exist_ok=True)
os.makedirs('models/xgboost', exist_ok=True)
os.makedirs('models/logistic_regression', exist_ok=True)
os.makedirs('results/tables', exist_ok=True)

joblib.dump(best_estimators['DecisionTreeClassifier'], 'models/decision_tree/model.joblib')
joblib.dump(best_estimators['RandomForestClassifier'], 'models/random_forest/model.joblib')
joblib.dump(best_estimators['XGBClassifier'], 'models/xgboost/model.joblib')
joblib.dump(best_estimators['LogisticRegression'], 'models/logistic_regression/model.joblib')
joblib.dump(voting_clf_soft, 'models/voting_ensemble_soft.joblib')
joblib.dump(voting_clf_hard, 'models/voting_ensemble_hard.joblib')

results_df = pd.DataFrame(list(test_results.items()), columns=['model', 'macro_f1'])
results_df = results_df.sort_values('macro_f1', ascending=False)
results_df.to_csv('results/tables/model_comparison.csv', index=False)
print(results_df)
print('\nBest model for SPR-05 SHAP analysis:', results_df.iloc[0]['model'])